# Data Processing with pandas

pandas is the standard library for working with tabular data in Python. A DataFrame is like a spreadsheet you can manipulate with code: filter rows, compute new columns, group and aggregate, and join tables, all in a few lines. If you've ever found yourself doing repetitive data work in Excel, pandas is the answer.

**What's inside:** creating DataFrames, inspecting data, selecting rows and columns with `loc`/`iloc`, boolean filtering, `apply()`, handling missing data, `groupby` aggregation, sorting, and merging.

**Learn more:** [pandas documentation](https://pandas.pydata.org/)

## Setup

In [ ]:
%pip install pandas

## 1. Creating DataFrames

### 1.1 From a dictionary

Each key becomes a column name; each list becomes the column values.

In [1]:
import pandas as pd

df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'age':   [24, 31, 22, 28, 35],
    'score': [88.5, 74.0, 95.0, 61.5, 82.0],
    'grade': ['B', 'C', 'A', 'D', 'B'],
    'city':  ['Austin', 'Boston', 'Austin', 'Denver', 'Boston'],
})
df

,name,age,score,grade,city
0,Alice,24,88.5,B,Austin
1,Bob,31,74.0,C,Boston
2,Carol,22,95.0,A,Austin
3,Dave,28,61.5,D,Denver
4,Eve,35,82.0,B,Boston


### 1.2 From a list of dicts

Each dict is one row. Missing keys become NaN.

In [2]:
# each dict is one row
rows = [{'item': 'apple', 'qty': 3}, {'item': 'banana', 'qty': 5}]
pd.DataFrame(rows)

,item,qty
0,apple,3
1,banana,5


### 1.3 From a CSV string

Useful for quick inline test data without a file.

In [3]:
from io import StringIO

csv = 'x,y\n1,10\n2,20\n3,30'
pd.read_csv(StringIO(csv))

,x,y
0,1,10
1,2,20
2,3,30


## 2. Inspecting DataFrames

### 2.1 head / tail

In [4]:
df.head(3)

,name,age,score,grade,city
0,Alice,24,88.5,B,Austin
1,Bob,31,74.0,C,Boston
2,Carol,22,95.0,A,Austin


In [5]:
df.tail(2)

,name,age,score,grade,city
3,Dave,28,61.5,D,Denver
4,Eve,35,82.0,B,Boston


### 2.2 shape and dtypes

In [6]:
df.shape   # (rows, columns)

(5, 5)

In [7]:
df.dtypes

name         str
age        int64
score    float64
grade        str
city         str
dtype: object

### 2.3 info

Shows column names, non-null counts, and dtypes in one call.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   name    5 non-null      str    
 1   age     5 non-null      int64  
 2   score   5 non-null      float64
 3   grade   5 non-null      str    
 4   city    5 non-null      str    
dtypes: float64(1), int64(1), str(3)
memory usage: 332.0 bytes


### 2.4 describe

Summary statistics for all numeric columns.

In [9]:
df.describe()

,age,score
count,5.000000,5.000000
mean,28.000000,80.200000
std,5.244044,13.031692
min,22.000000,61.500000
25%,24.000000,74.000000
50%,28.000000,82.000000
75%,31.000000,88.500000
max,35.000000,95.000000


## 3. Selecting Data

### 3.1 Single column

In [10]:
df['score']

0    88.5
1    74.0
2    95.0
3    61.5
4    82.0
Name: score, dtype: float64

### 3.2 Multiple columns

In [11]:
df[['name', 'score']]

,name,score
0,Alice,88.5
1,Bob,74.0
2,Carol,95.0
3,Dave,61.5
4,Eve,82.0


### 3.3 loc: label-based selection

`loc[row_label, column_name]`: rows and columns by their names/index labels.

In [12]:
# single value by row label and column name
df.loc[2, 'name']

'Carol'

In [13]:
# rows 1 through 3, selected columns
df.loc[1:3, ['name', 'score']]

,name,score
1,Bob,74.0
2,Carol,95.0
3,Dave,61.5


### 3.4 iloc: position-based selection

`iloc[row_index, col_index]`: rows and columns by integer position.

In [14]:
# last two rows, first two columns
df.iloc[-2:, :2]

,name,age
3,Dave,28
4,Eve,35


## 4. Boolean Filtering

### 4.1 Single condition

In [15]:
df[df['score'] > 80]

,name,age,score,grade,city
0,Alice,24,88.5,B,Austin
2,Carol,22,95.0,A,Austin
4,Eve,35,82.0,B,Boston


### 4.2 Multiple conditions

Use `&` for AND, `|` for OR; each condition must be wrapped in parentheses.

In [16]:
df[(df['score'] > 70) & (df['city'] == 'Boston')]

,name,age,score,grade,city
1,Bob,31,74.0,C,Boston
4,Eve,35,82.0,B,Boston


### 4.3 isin

Filter rows where a column's value is in a set.

In [17]:
df[df['grade'].isin(['A', 'B'])]

,name,age,score,grade,city
0,Alice,24,88.5,B,Austin
2,Carol,22,95.0,A,Austin
4,Eve,35,82.0,B,Boston


### 4.4 query()

Write filter conditions as a readable string, especially useful for complex filters.

In [18]:
df.query('score > 80 and city == "Austin"')

,name,age,score,grade,city
0,Alice,24,88.5,B,Austin
2,Carol,22,95.0,A,Austin


## 5. Adding and Modifying Columns

### 5.1 New column from arithmetic

In [19]:
df['score_pct'] = df['score'] / 100
df[['name', 'score', 'score_pct']].head(3)

,name,score,score_pct
0,Alice,88.5,0.885
1,Bob,74.0,0.740
2,Carol,95.0,0.950


### 5.2 apply() with a lambda

In [20]:
# classify each score as pass or fail
df['result'] = df['score'].apply(lambda x: 'pass' if x >= 70 else 'fail')
df[['name', 'score', 'result']]

,name,score,result
0,Alice,88.5,pass
1,Bob,74.0,pass
2,Carol,95.0,pass
3,Dave,61.5,fail
4,Eve,82.0,pass


### 5.3 apply() with a named function

In [21]:
def age_group(age):
    return 'young' if age < 30 else 'senior'

df['age_group'] = df['age'].apply(age_group)
df[['name', 'age', 'age_group']]

,name,age,age_group
0,Alice,24,young
1,Bob,31,senior
2,Carol,22,young
3,Dave,28,young
4,Eve,35,senior


### 5.4 apply() across rows with axis=1

`axis=1` passes each row as a Series, letting you access multiple columns per row.

In [22]:
df['label'] = df.apply(lambda r: r['name'] + ' (' + r['city'] + ')', axis=1)
df['label']

0    Alice (Austin)
1      Bob (Boston)
2    Carol (Austin)
3     Dave (Denver)
4      Eve (Boston)
Name: label, dtype: str

## 6. Handling Missing Data

In [23]:
messy = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol'],
    'score': [88.5, None, 95.0],
    'city':  ['Austin', 'Boston', None],
})
messy

,name,score,city
0,Alice,88.5,Austin
1,Bob,NaN,Boston
2,Carol,95.0,NaN


### 6.1 Detecting nulls

In [24]:
messy.isna().sum()   # null count per column

name     0
score    1
city     1
dtype: int64

### 6.2 dropna

In [25]:
messy.dropna()   # drop any row with at least one null

,name,score,city
0,Alice,88.5,Austin


In [26]:
messy.dropna(subset=['score'])   # only drop rows where 'score' is null

,name,score,city
0,Alice,88.5,Austin
2,Carol,95.0,NaN


### 6.3 fillna

In [27]:
messy.fillna({'score': 0, 'city': 'Unknown'})

,name,score,city
0,Alice,88.5,Austin
1,Bob,0.0,Boston
2,Carol,95.0,Unknown


## 7. groupby and Aggregation

### 7.1 Single aggregation

In [28]:
df.groupby('city')['score'].mean()

city
Austin    91.75
Boston    78.00
Denver    61.50
Name: score, dtype: float64

### 7.2 Multiple aggregations with agg()

In [29]:
df.groupby('city')['score'].agg(['mean', 'max', 'count'])

,mean,max,count
city,,,
Austin,91.75,95.0,2
Boston,78.00,82.0,2
Denver,61.50,61.5,1


### 7.3 Multiple columns

In [30]:
df.groupby('city')[['age', 'score']].mean()

,age,score
city,,
Austin,23.0,91.75
Boston,33.0,78.00
Denver,28.0,61.50


## 8. Sorting

In [31]:
df.sort_values('score', ascending=False)[['name', 'score']]

,name,score
2,Carol,95.0
0,Alice,88.5
4,Eve,82.0
1,Bob,74.0
3,Dave,61.5


In [32]:
# sort by multiple columns (city ascending, score descending within each city)
df.sort_values(['city', 'score'], ascending=[True, False])[['name', 'city', 'score']]

,name,city,score
2,Carol,Austin,95.0
0,Alice,Austin,88.5
4,Eve,Boston,82.0
1,Bob,Boston,74.0
3,Dave,Denver,61.5


## 9. Merging DataFrames

In [33]:
# a second table referencing student names
sales = pd.DataFrame({
    'name':   ['Alice', 'Bob', 'Alice', 'Carol'],
    'amount': [200, 150, 300, 100],
})

### 9.1 Inner join (default)

Keep only rows with matching keys in both tables.

In [34]:
pd.merge(df[['name', 'city']], sales, on='name')

,name,city,amount
0,Alice,Austin,200
1,Alice,Austin,300
2,Bob,Boston,150
3,Carol,Austin,100


### 9.2 Left join

Keep all rows from the left table; NaN where the right table has no match.

In [35]:
pd.merge(df[['name', 'score']], sales, on='name', how='left')

,name,score,amount
0,Alice,88.5,200.0
1,Alice,88.5,300.0
2,Bob,74.0,150.0
3,Carol,95.0,100.0
4,Dave,61.5,NaN
5,Eve,82.0,NaN
